# Regresión Logística con Clases Balanceadas


In [ ]:
# Librerías esenciales para análisis de datos y modelado
import pandas as pd
import numpy as np
from sklearn import preprocessing
import matplotlib.pyplot as plt 
plt.rc("font", size=14)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import seaborn as sns
sns.set(style="white")
sns.set(style="whitegrid", color_codes=True)

## Los Datos

Los datos provienen de campañas de marketing directo (llamadas telefónicas) de una institución bancaria portuguesa. El objetivo de clasificación es predecir si el cliente suscribirá (1/0) un depósito a plazo (variable y).

Este conjunto proporciona información del cliente. Incluye aproximadamente 41.000 registros y 21 campos.



In [ ]:
# Carga del dataset bank-additional-full.csv (separador ;)
# Conversión de y: yes->1, no->0 para compatibilidad con el modelo
# Renombrado de columnas con puntos para evitar conflictos en Python
data = pd.read_csv('bank-additional-full.csv', header=0, sep=';')
data = data.dropna()

# Mapear variable objetivo: yes=1 (suscripción), no=0 (no suscripción)
data['y'] = data['y'].map({'yes': 1, 'no': 0})

# Renombrar columnas con puntos (formato UCI original)
data = data.rename(columns={
    'emp.var.rate': 'emp_var_rate',
    'cons.price.idx': 'cons_price_idx', 
    'cons.conf.idx': 'cons_conf_idx',
    'nr.employed': 'nr_employed'
})

print("Dimensiones del dataset:", data.shape)
print("Columnas:", list(data.columns))

In [ ]:
# Primeras filas del dataset
data.head()

#### Variables de entrada

1 - age (numérica)
2 - job: tipo de empleo (categórica)
3 - marital: estado civil (categórica)
4 - education: nivel educativo (categórica)
5 - default: ¿tiene crédito en mora? (categórica)
6 - housing: ¿tiene préstamo hipotecario? (categórica)
7 - loan: ¿tiene préstamo personal? (categórica)
8 - contact: tipo de contacto (categórica)
9 - month: último mes de contacto (categórica)
10 - day_of_week: último día de contacto (categórica)
11 - duration: duración del último contacto en segundos (numérica). **Nota:** Esta variable afecta fuertemente el target; no debe usarse en modelos predictivos realistas.
12 - campaign: número de contactos en esta campaña (numérica)
13 - pdays: días desde último contacto de campaña anterior (999 = no contactado)
14 - previous: contactos previos (numérica)
15 - poutcome: resultado campaña anterior (categórica)
16-20 - Indicadores macroeconómicos (emp_var_rate, cons_price_idx, etc.)

#### Variable objetivo: y - ¿El cliente suscribió un depósito a plazo? (binaria: 1/0)

**Comentario:** La columna education tiene muchas categorías; agruparemos basic.4y, basic.6y y basic.9y en "Basic" para reducir dimensionalidad y mejorar la generalización.

In [ ]:
# Categorías originales en education
data['education'].unique()

Agrupando "basic.4y", "basic.9y" y "basic.6y" en una sola categoría "Basic".

In [ ]:
# Agrupación de niveles básicos de educación
data['education'] = np.where(data['education'] == 'basic.9y', 'Basic', data['education'])
data['education'] = np.where(data['education'] == 'basic.6y', 'Basic', data['education'])
data['education'] = np.where(data['education'] == 'basic.4y', 'Basic', data['education'])

Categorías de educación después del agrupamiento:

In [ ]:
data['education'].unique()

### Exploración de datos

### Análisis descriptivo univariado

En el análisis univariado se examina cada variable por separado para entender su distribución, tendencia central y dispersión antes de relacionarla con la variable respuesta.

In [ ]:
# Variables numéricas: estadísticos descriptivos (media, mediana, desv. estándar, cuartiles)
vars_num = ['age', 'duration', 'campaign', 'pdays', 'previous', 'emp_var_rate', 
            'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed']
print("=== Estadísticos descriptivos (variables numéricas) ===\n")
data[vars_num].describe()

In [ ]:
# Histogramas de variables numéricas clave
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, var in enumerate(['age', 'duration', 'campaign', 'pdays', 'previous', 'euribor3m']):
    data[var].hist(ax=axes[i], bins=30, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'Distribución de {var}')
    axes[i].set_xlabel(var)
plt.tight_layout()
plt.suptitle('Análisis univariado: distribuciones de variables numéricas', y=1.02, fontsize=12)
plt.show()

In [ ]:
# Variables numéricas: diagramas de caja (boxplot) para detectar outliers y asimetría
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, var in enumerate(['age', 'duration', 'campaign', 'pdays', 'previous', 'euribor3m']):
    data.boxplot(column=var, ax=axes[i])
    axes[i].set_title(f'Boxplot de {var}')
plt.tight_layout()
plt.suptitle('Análisis univariado: boxplots (valores atípicos y dispersión)', y=1.02, fontsize=12)
plt.show()

In [ ]:
# Variables categóricas: frecuencias absolutas y relativas
vars_cat = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']
print("=== Frecuencias univariadas (variables categóricas) ===\n")
for var in vars_cat:
    print(f"\n--- {var} ---")
    freq = data[var].value_counts()
    pct = data[var].value_counts(normalize=True).round(4) * 100
    df_freq = pd.DataFrame({'Frecuencia': freq, 'Porcentaje (%)': pct})
    print(df_freq)

In [ ]:
# Gráficos de barras para variables categóricas principales
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, var in zip(axes.flatten(), ['job', 'marital', 'education', 'poutcome']):
    data[var].value_counts().plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
    ax.set_title(f'Distribución de {var}')
    ax.set_xlabel(var)
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.suptitle('Análisis univariado: distribuciones de variables categóricas', y=1.02, fontsize=12)
plt.show()

### Análisis descriptivo bivariado con la variable respuesta (y)

Se explora la relación entre cada predictor y la variable objetivo `y` (1 = suscripción, 0 = no suscripción) mediante:
- **Numéricas vs y:** boxplots por grupo, medias por grupo y pruebas t cuando proceda.
- **Categóricas vs y:** tablas cruzadas, proporciones condicionales, chi-cuadrado y gráficos de barras apiladas/agrupadas.

In [ ]:
# Medias por grupo (y=0 vs y=1) para variables numéricas
print("=== Medias por grupo (No suscripción vs Suscripción) ===\n")
medias_por_grupo = data.groupby('y')[vars_num].mean().T
medias_por_grupo.columns = ['No suscripción (y=0)', 'Suscripción (y=1)']
medias_por_grupo['Diferencia'] = medias_por_grupo['Suscripción (y=1)'] - medias_por_grupo['No suscripción (y=0)']
medias_por_grupo.round(4)

In [ ]:
# Boxplots bivariados: variables numéricas vs y
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, var in enumerate(['age', 'duration', 'campaign', 'pdays', 'previous', 'euribor3m']):
    plot_data = [data[data['y']==0][var].values, data[data['y']==1][var].values]
    axes[i].boxplot(plot_data, labels=['No (0)', 'Sí (1)'])
    axes[i].set_title(f'{var} por suscripción')
    axes[i].set_ylabel(var)
plt.suptitle('Análisis bivariado: variables numéricas vs variable respuesta', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Prueba t de Student para comparar medias (variables numéricas vs y)
from scipy import stats
print("=== Prueba t: diferencia de medias entre y=0 e y=1 ===\n")
for var in vars_num:
    g0 = data[data['y']==0][var].dropna()
    g1 = data[data['y']==1][var].dropna()
    t_stat, p_val = stats.ttest_ind(g0, g1)
    print(f"{var}: t={t_stat:.4f}, p-valor={p_val:.4e} {'***' if p_val<0.001 else '**' if p_val<0.01 else '*' if p_val<0.05 else ''}")

In [ ]:
# Tablas cruzadas y proporciones condicionales (categóricas vs y)
print("=== Tablas cruzadas y % de suscripción por categoría ===\n")
for var in vars_cat:
    tabla = pd.crosstab(data[var], data['y'], margins=True)
    tabla_pct = pd.crosstab(data[var], data['y'], normalize='index') * 100
    print(f"\n--- {var} ---")
    print("Frecuencias:")
    print(tabla)
    print("\nPorcentaje por fila (col 1 = % suscripción):")
    print(tabla_pct.round(2))

In [ ]:
# Chi-cuadrado de independencia (categóricas vs y)
print("=== Chi-cuadrado: asociación entre variables categóricas y y ===\n")
for var in vars_cat:
    tabla = pd.crosstab(data[var], data['y'])
    chi2, p_val, dof, expected = stats.chi2_contingency(tabla)
    print(f"{var}: chi2={chi2:.2f}, p-valor={p_val:.4e} {'***' if p_val<0.001 else '**' if p_val<0.01 else '*' if p_val<0.05 else ''}")

In [ ]:
# Gráficos de barras apiladas: categóricas vs y
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, var in zip(axes.flatten(), ['job', 'marital', 'education', 'poutcome']):
    pd.crosstab(data[var], data['y']).plot(kind='bar', stacked=True, ax=ax, 
        color=['#2ecc71', '#e74c3c'], edgecolor='black')
    ax.set_title(f'{var} vs suscripción (y)')
    ax.set_xlabel(var)
    ax.legend(['No (0)', 'Sí (1)'])
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.suptitle('Análisis bivariado: variables categóricas vs variable respuesta', y=1.02, fontsize=12)
plt.show()

In [ ]:
# Gráficos de barras agrupadas: % de suscripción por categoría (para comparar tasas)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, var in zip(axes.flatten(), ['job', 'marital', 'education', 'poutcome']):
    pct_si = data.groupby(var)['y'].mean() * 100
    pct_si.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
    ax.set_title(f'% Suscripción por {var}')
    ax.set_ylabel('% Suscripción (y=1)')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.suptitle('Tasa de suscripción por categoría', y=1.02, fontsize=12)
plt.show()

In [ ]:
# Conteo por clase: desbalance crítico para el modelado
# Comentario: Un ratio 89:11 implica que un modelo "naive" que prediga siempre "no" 
# alcanzaría ~89% de precisión sin aprender nada. SMOTE mitiga este problema.
data['y'].value_counts()

In [ ]:
# Visualización del desbalance de clases
sns.countplot(x='y', data=data, palette='hls')
plt.title('Distribución de la variable objetivo (y)')
plt.show()

In [ ]:
# Porcentajes de cada clase
count_no_sub = len(data[data['y']==0])
count_sub = len(data[data['y']==1])
pct_of_no_sub = count_no_sub/(count_no_sub+count_sub)
print("Porcentaje de no suscripción:", round(pct_of_no_sub*100, 2), "%")
pct_of_sub = count_sub/(count_no_sub+count_sub)
print("Porcentaje de suscripción:", round(pct_of_sub*100, 2), "%")

Las clases están desbalanceadas; el ratio no-suscripción:suscripción es aproximadamente 89:11. Antes de balancear, exploramos más los datos.

In [ ]:
# Medias por grupo (suscritos vs no suscritos)
# Comentario: Las diferencias en duration, pdays y campaign son estadísticamente 
# relevantes. Los suscritos tienen mayor duración de llamada (más tiempo escuchando la oferta)
# y menor pdays (contacto más reciente = mejor memoria de la oferta).
data.groupby('y').mean()

**Observaciones:**
- La edad media de quienes suscribieron es ligeramente mayor.
- pdays (días desde último contacto) es menor en suscritos: contacto más reciente mejora la conversión.
- Las campañas (número de contactos) son menores en suscritos: menos intentos necesarios para convencer.
- Podemos calcular medias por variables categóricas (job, education, marital) para un análisis más detallado.

In [ ]:
# Medias por tipo de empleo - la columna y muestra la tasa de conversión por job
data.groupby('job').mean()

In [ ]:
# Medias por estado civil
data.groupby('marital').mean()

In [ ]:
# Medias por nivel educativo
data.groupby('education').mean()

### Visualizaciones

In [ ]:
# Frecuencia de suscripción por tipo de empleo
# Comentario: retired y student muestran tasas más altas; blue-collar la más baja.
# Esto sugiere que job es un predictor útil para el modelo.
pd.crosstab(data.job, data.y).plot(kind='bar')
plt.title('Frecuencia de suscripción por tipo de empleo')
plt.xlabel('Empleo')
plt.ylabel('Frecuencia de suscripción')
plt.legend(['No suscripción', 'Suscripción'])
plt.show()

In [ ]:
# Gráfico apilado: estado civil vs suscripción
table = pd.crosstab(data.marital, data.y)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Proporción de suscripción por estado civil')
plt.xlabel('Estado civil')
plt.ylabel('Proporción de clientes')
plt.legend(['No suscripción', 'Suscripción'])
plt.show()

In [ ]:
# Proporción de suscripción por educación
table = pd.crosstab(data.education, data.y)
table.div(table.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Proporción de suscripción por educación')
plt.xlabel('Educación')
plt.ylabel('Proporción de clientes')
plt.legend(['No suscripción', 'Suscripción'])
plt.show()

In [ ]:
# Frecuencia por día de la semana
pd.crosstab(data.day_of_week, data.y).plot(kind='bar')
plt.title('Frecuencia de suscripción por día de la semana')
plt.xlabel('Día de la semana')
plt.ylabel('Frecuencia')
plt.legend(['No suscripción', 'Suscripción'])
plt.show()

In [ ]:
# Frecuencia por mes
pd.crosstab(data.month, data.y).plot(kind='bar')
plt.title('Frecuencia de suscripción por mes')
plt.xlabel('Mes')
plt.ylabel('Frecuencia')
plt.legend(['No suscripción', 'Suscripción'])
plt.show()

In [ ]:
# Histograma de edad
# Comentario: La mayoría de clientes están entre 30-40 años.
data.age.hist()
plt.title('Histograma de edad')
plt.xlabel('Edad')
plt.ylabel('Frecuencia')
plt.show()

In [ ]:
# Resultado de campaña anterior vs suscripción
# Comentario: poutcome='success' correlaciona con mayor tasa de conversión.
pd.crosstab(data.poutcome, data.y).plot(kind='bar')
plt.title('Frecuencia de suscripción por resultado campaña anterior')
plt.xlabel('Resultado anterior')
plt.ylabel('Frecuencia')
plt.legend(['No suscripción', 'Suscripción'])
plt.show()

### Creación de variables dummy

**Comentario:** Las variables categóricas deben codificarse para regresión logística. Usamos one-hot encoding (get_dummies) para evitar orden artificial en categorías nominales.

In [ ]:
# One-hot encoding de variables categóricas
cat_vars = ['job','marital','education','default','housing','loan','contact','month','day_of_week','poutcome']
for var in cat_vars:
    cat_list = pd.get_dummies(data[var], prefix=var)
    data = data.join(cat_list)
    
data_vars = data.columns.values.tolist()
to_keep = [i for i in data_vars if i not in cat_vars]
data_final = data[to_keep]
data_final.columns.values

### Sobremuestreo con SMOTE

**Comentario:** SMOTE (Synthetic Minority Over-sampling Technique) genera ejemplos sintéticos de la clase minoritaria interpolando entre vecinos cercanos. Esto equilibra las clases sin duplicar exactamente los mismos registros, reduciendo el sobreajuste que causaría un simple oversampling por repetición.

In [ ]:
# Preparación de X e y
X = data_final.loc[:, data_final.columns != 'y']
y = data_final.loc[:, data_final.columns == 'y']

In [ ]:
# Aplicación de SMOTE para balancear clases
from imblearn.over_sampling import SMOTE

os = SMOTE(random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
columns = X_train.columns

# fit_resample es el método actual en imblearn
try:
    os_data_X, os_data_y = os.fit_resample(X_train, y_train.values.ravel())
except AttributeError:
    os_data_X, os_data_y = os.fit_sample(X_train, y_train.values.ravel())

os_data_X = pd.DataFrame(data=os_data_X, columns=columns)
os_data_y = pd.DataFrame(data=os_data_y, columns=['y'])

print("Tamaño del dataset balanceado:", len(os_data_X))
print("No suscripciones:", len(os_data_y[os_data_y['y']==0]))
print("Suscripciones:", len(os_data_y[os_data_y['y']==1]))
print("Proporción clase 0:", round(len(os_data_y[os_data_y['y']==0])/len(os_data_X), 2))
print("Proporción clase 1:", round(len(os_data_y[os_data_y['y']==1])/len(os_data_X), 2))

### Eliminación recursiva de características (RFE)

**Comentario:** RFE selecciona las features más relevantes entrenando el modelo y eliminando iterativamente las menos importantes. Reduce el riesgo de multicollinearity y mejora la interpretabilidad.

In [ ]:
# RFE para selección de 20 features más relevantes
from sklearn.feature_selection import RFE

logreg = LogisticRegression(max_iter=1000)
rfe = RFE(logreg, n_features_to_select=20)
rfe = rfe.fit(os_data_X, os_data_y.values.ravel())
print("Features seleccionadas:", rfe.support_)
print("Ranking:", rfe.ranking_)

RFE seleccionó features como: euribor3m, job_blue-collar, job_housemaid, marital_unknown, education_illiterate, default_no, month_*, poutcome_*, etc.

In [ ]:
# Subconjunto de features para el modelo (basado en RFE y significancia estadística)
cols = ['euribor3m', 'job_blue-collar', 'job_housemaid', 'marital_unknown', 'education_illiterate', 
        'month_apr', 'month_aug', 'month_dec', 'month_jul', 'month_jun', 'month_mar', 
        'month_may', 'month_nov', 'month_oct', 'poutcome_failure', 'poutcome_success'] 
# Eliminar columnas que no existan (por variaciones en el dataset)
cols = [c for c in cols if c in os_data_X.columns]
X = os_data_X[cols]
y = os_data_y['y']

### Ajuste con statsmodels (análisis de significancia)

**Comentario:** statsmodels proporciona p-valores y estadísticos detallados. Si p>0.05, la variable no es estadísticamente significativa y puede eliminarse.

In [ ]:
# Modelo Logit con statsmodels para verificación de coeficientes
import statsmodels.api as sm

logit_model = sm.Logit(y, X)
result = logit_model.fit()
print(result.summary2())

### Entrenamiento del modelo con sklearn

In [ ]:
# Entrenamiento de regresión logística
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)

In [ ]:
# Precisión en el conjunto de prueba
# Comentario: 74% es razonable para datos con ruido y alto desbalance original.
# La métrica de accuracy puede ser engañosa; revisar matriz de confusión y ROC.
y_pred = logreg.predict(X_test)
print('Precisión del clasificador en conjunto de prueba: {:.2f}'.format(logreg.score(X_test, y_test)))

### Matriz de confusión

**Comentario:** La matriz muestra: TN (esq. sup. izq.), FP (sup. der.), FN (inf. izq.), TP (inf. der.). Para marketing, el recall de la clase positiva (suscripción) es crucial: no queremos perder clientes potenciales.

In [ ]:
# Matriz de confusión
from sklearn.metrics import confusion_matrix
confusion_matrix = confusion_matrix(y_test, y_pred)
print(confusion_matrix)
print("Interpretación: [TN, FP] / [FN, TP]")

In [ ]:
# Reporte de clasificación: precision, recall, F1 por clase
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

### Curva ROC

**Comentario:** El área bajo la curva (AUC) mide la capacidad del modelo para distinguir entre clases. AUC=0.5 es aleatorio; AUC>0.7 se considera aceptable; AUC>0.8 es bueno.

In [ ]:
# Curva ROC y AUC
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve

logit_roc_auc = roc_auc_score(y_test, logreg.predict(X_test))
fpr, tpr, thresholds = roc_curve(y_test, logreg.predict_proba(X_test)[:,1])
plt.figure()
plt.plot(fpr, tpr, label='Regresión Logística (AUC = %0.2f)' % logit_roc_auc)
plt.plot([0, 1], [0, 1], 'r--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de falsos positivos')
plt.ylabel('Tasa de verdaderos positivos')
plt.title('Característica operativa del receptor (ROC)')
plt.legend(loc="lower right")
plt.savefig('Log_ROC.png')
plt.show()

## Resumen

- **Dataset:** Bank Marketing (UCI), campañas de depósitos a plazo
- **Problema:** Clasificación binaria con clases desbalanceadas (89:11)
- **Solución:** SMOTE para balanceo + RFE para selección de features
- **Modelo:** Regresión logística con ~74% de precisión
- **Interpretación:** Variables como euribor3m, poutcome_success, mes y tipo de empleo son predictores clave.